# FoodTrip — train a PhoBERT NER model to replace the `detect-place` Anthropic call

Task: given a video caption (and later, OCR'd thumbnail text), extract `BUSINESS` and `LOC` entity spans — the same job the Anthropic API currently does in `supabase/functions/detect-place`.

**Data**: `ml/ner-dataset/{train,val,test}.jsonl`. Two ways to generate it:
- `node scripts/generate-ner-dataset.js` — fully synthetic, from the app's fictional 59-place seed dataset. Zero setup, good for a first training run.
- `node scripts/crawl-real-places.js` (needs `GOOGLE_PLACES_SERVER_KEY`) + `node scripts/crawl-youtube-captions.js` (needs `YOUTUBE_API_KEY`) + `node scripts/build-real-ner-dataset.js` — pulls real business names from Google Places and real YouTube captions, auto-labels the captions by matching against the real place list, and writes a dataset grounded in real data instead of fictional entities. TikTok has no public search API, so YouTube is the only platform crawled in bulk here.

Upload the three `.jsonl` files (and `labels.json`) from `ml/ner-dataset/` in your repo to this Colab session before running — whichever way you generated them.

**Runtime**: Runtime → Change runtime type → GPU (T4 is enough for PhoBERT-base).

In [ ]:
!pip install -q transformers datasets seqeval accelerate optimum[exporters]

## 1. Upload the dataset
Run this cell, then pick `train.jsonl`, `val.jsonl`, `test.jsonl`, `labels.json` from `ml/ner-dataset/` in your local repo.

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.replace(name, f'data/{name}')
print(os.listdir('data'))

## 2. Load labels + dataset

In [ ]:
import json
from datasets import load_dataset, ClassLabel, Sequence, Features, Value

with open('data/labels.json') as f:
    label_list = json.load(f)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
print(label_list)

raw = load_dataset('json', data_files={
    'train': 'data/train.jsonl',
    'validation': 'data/val.jsonl',
    'test': 'data/test.jsonl',
})

def encode_tags(example):
    example['ner_tags'] = [label2id[t] for t in example['ner_tags']]
    return example

raw = raw.map(encode_tags)
raw['train'][0]

## 3. Tokenize with PhoBERT + align labels to subwords

PhoBERT expects **word-segmented** Vietnamese (words joined with `_`, e.g. `Hội_An`). The synthetic dataset's `tokens` are plain words (no compound segmentation) — this is a known simplification documented in `scripts/generate-ner-dataset.js`. For a stronger model, re-segment `tokens` with `underthesea.word_tokenize` before this step; the notebook works either way, just with lower accuracy on multi-syllable place names if you skip it.

Install `underthesea` and segment now (recommended):

In [ ]:
!pip install -q underthesea

In [ ]:
from underthesea import word_tokenize

def resegment(example):
    # Re-join original tokens to text, then re-segment + re-align tags by
    # matching each new (possibly compound, '_'-joined) token back to the
    # span of original tokens it covers, and keeping the first covered tag.
    tokens, tags = example['tokens'], example['ner_tags']
    new_tokens, new_tags = [], []
    i = 0
    text = ' '.join(tokens)
    segmented = word_tokenize(text)
    orig_pos = 0
    for seg in segmented:
        parts = seg.split('_')
        # Best-effort alignment: take the tag of the first original token
        # this segment starts at, then advance the cursor by len(parts).
        tag = tags[orig_pos] if orig_pos < len(tags) else 0
        new_tokens.append(seg)
        new_tags.append(tag)
        orig_pos += len(parts)
    example['tokens'] = new_tokens
    example['ner_tags'] = new_tags
    return example

raw = raw.map(resegment)
raw['train'][0]

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = 'vinai/phobert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(labels[word_id])
            else:
                # subword continuation — use I-X of the same entity, or -100 if it was O
                base = labels[word_id]
                label_ids.append(base if id2label[base] == 'O' else base)
            prev_word_id = word_id
        all_labels.append(label_ids)
        prev_word_id = word_id
    tokenized['labels'] = all_labels
    return tokenized

tokenized_ds = raw.map(tokenize_and_align_labels, batched=True)

## 4. Fine-tune

In [ ]:
import numpy as np
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

seqeval = evaluate.load('seqeval')

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(label_list), id2label=id2label, label2id=label2id
)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_predictions, true_labels = [], []
    for pred, label in zip(predictions, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred, label):
            if l == -100:
                continue
            p_seq.append(id2label[p])
            l_seq.append(id2label[l])
        true_predictions.append(p_seq)
        true_labels.append(l_seq)
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

args = TrainingArguments(
    output_dir='phobert-ner-place',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Evaluate on the held-out test set

In [ ]:
trainer.evaluate(tokenized_ds['test'])

## 6. Quick manual test

In [ ]:
from transformers import pipeline

ner = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy='simple')
ner('Đi Hội An nhớ ghé Cao Lầu Bà Bé nha mọi người, ngon lắm')

## 7. Export to ONNX (for `onnxruntime-web` / edge-function inference — no API key needed)

In [ ]:
!optimum-cli export onnx --model phobert-ner-place --task token-classification phobert-ner-onnx/

from google.colab import files
import shutil
shutil.make_archive('phobert-ner-onnx', 'zip', 'phobert-ner-onnx')
files.download('phobert-ner-onnx.zip')

## Next steps back in the FoodTrip repo
1. Unzip `phobert-ner-onnx.zip` into e.g. `ml/models/phobert-ner-onnx/`.
2. Replace the Anthropic call in `supabase/functions/detect-place/index.ts` with either:
   - an `onnxruntime-node` inference call inside the edge function (keeps the same request/response shape), or
   - `onnxruntime-web` running client-side, skipping the edge function for this step entirely.
3. Keep `verify-place` (Google Places lookup) unchanged — that step is deterministic lookup, not something to train.
4. Once real `video_reviews` data accumulates, mix real labeled captions into `ml/ner-dataset/` and re-run this notebook — synthetic data should be a bootstrap, not the permanent training set.